In [7]:
!pip install -q google-cloud-bigquery pandas pandas-gbq pyarrow db-dtypes

from google.colab import auth, drive
from google.cloud import bigquery
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

auth.authenticate_user()
drive.mount("/content/drive")

PROJECT_ID = "enares-2024-crs04"
LOCATION = "US"
EXPECTED_ROWS = 18807

ROOT_DRIVE = Path(
    "/content/drive/MyDrive/ENARES_2024_PROJECT"
)

LOG_DIR = ROOT_DRIVE / "05Resultados" / "logs" / "stage03"
SQL_DIR = ROOT_DRIVE / "02SQL"
DOCS_DIR = ROOT_DRIVE / "docs"

for directory in [LOG_DIR, SQL_DIR, DOCS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RUN_UTC = datetime.now(timezone.utc).isoformat()

client = bigquery.Client(
    project=PROJECT_ID,
    location=LOCATION,
)

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

table = client.get_table(A)

if table.num_rows != EXPECTED_ROWS:
    raise RuntimeError(
        f"Analytical tiene {table.num_rows} filas; "
        f"se esperaban {EXPECTED_ROWS}."
    )

required_previous = [
    "VP_ESCUELA",
    "VF_ESCUELA",
    "VP_o_VF_ESCUELA",
    "VP_VF_ESCUELA",
    "VS_ESCUELA",
    "INDICADOR_8_3_9",
]

existing = {field.name for field in table.schema}

missing_previous = sorted(
    set(required_previous) - existing
)

if missing_previous:
    raise RuntimeError(
        "Ejecuta primero el notebook 03. Faltan: "
        + ", ".join(missing_previous)
    )

print(
    "Prerequisito aprobado: analytical existe, "
    "tiene 18,807 filas y contiene el bloque 3.3."
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Prerequisito aprobado: analytical existe, tiene 18,807 filas y contiene el bloque 3.3.


In [8]:
# ============================================================
# SPSS 3.4 — VS_12M
# Violencia sexual durante los últimos 12 meses
# ============================================================

existing = {
    field.name
    for field in client.get_table(A).schema
}

required_vs12 = []

for i in range(1, 17):
    required_vs12.extend([
        f"C4P248_{i}",
        f"C4P248C_{i}",
    ])

missing_vs12 = sorted(
    set(required_vs12) - existing
)

if missing_vs12:
    raise RuntimeError(
        "No se puede crear VS_12M. Faltan: "
        + ", ".join(missing_vs12)
    )

conditions_vs12 = [
    f"(`C4P248_{i}` = 1 AND `C4P248C_{i}` = 1)"
    for i in range(1, 17)
]

any_vs12 = "\nOR\n".join(conditions_vs12)

select_prefix = (
    "* EXCEPT(VS_12M)"
    if "VS_12M" in existing
    else "*"
)

vs12_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
  {select_prefix},

  CASE
    WHEN (
      {any_vs12}
    )
    THEN 1
    ELSE 0
  END AS VS_12M

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_34_vs12m.sql").write_text(
    vs12_sql,
    encoding="utf-8",
)

client.query(vs12_sql).result()

print("VS_12M creado correctamente.")

VS_12M creado correctamente.


In [9]:
# ============================================================
# SPSS 3.4 — Formas ICVAC, últimos 12 meses
# ============================================================

derived_icvac = [
    "VS_ICVAC_301",
    "VS_ICVAC_302",
    "VS_ICVAC_303",
    "VS_ICVAC_309",
]

existing = {
    field.name
    for field in client.get_table(A).schema
}

drop_icvac = [
    column
    for column in derived_icvac
    if column in existing
]

source_select = (
    "* EXCEPT("
    + ", ".join(f"`{column}`" for column in drop_icvac)
    + ")"
    if drop_icvac
    else "*"
)

icvac_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
  {source_select},

  CASE
    WHEN C4P248_11 = 1
     AND C4P248C_11 = 1
    THEN 1
    ELSE 0
  END AS VS_ICVAC_301,

  CASE
    WHEN
      (C4P248_4 = 1 AND C4P248C_4 = 1)
      OR (C4P248_5 = 1 AND C4P248C_5 = 1)
      OR (C4P248_6 = 1 AND C4P248C_6 = 1)
    THEN 1
    ELSE 0
  END AS VS_ICVAC_302,

  CASE
    WHEN
      (C4P248_1 = 1 AND C4P248C_1 = 1)
      OR (C4P248_2 = 1 AND C4P248C_2 = 1)
      OR (C4P248_3 = 1 AND C4P248C_3 = 1)
      OR (C4P248_7 = 1 AND C4P248C_7 = 1)
      OR (C4P248_8 = 1 AND C4P248C_8 = 1)
      OR (C4P248_9 = 1 AND C4P248C_9 = 1)
      OR (C4P248_10 = 1 AND C4P248C_10 = 1)
      OR (C4P248_13 = 1 AND C4P248C_13 = 1)
      OR (C4P248_14 = 1 AND C4P248C_14 = 1)
      OR (C4P248_15 = 1 AND C4P248C_15 = 1)
      OR (C4P248_16 = 1 AND C4P248C_16 = 1)
    THEN 1
    ELSE 0
  END AS VS_ICVAC_303,

  CASE
    WHEN C4P248_12 = 1
     AND C4P248C_12 = 1
    THEN 1
    ELSE 0
  END AS VS_ICVAC_309

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_34_icvac.sql").write_text(
    icvac_sql,
    encoding="utf-8",
)

client.query(icvac_sql).result()

print("Formas ICVAC creadas correctamente.")

Formas ICVAC creadas correctamente.


In [10]:
# ============================================================
# SPSS 3.4 — Categorías analíticas del Código Penal
# ============================================================

cp_columns = [
    "VS_CP_ACOSO_AGRAVADO",
    "VS_CP_EXHIBICIONISMO",
    "VS_CP_TOCAMIENTOS",
    "VS_CP_TENTATIVA_VIOLACION",
    "VS_CP_VIOLACION_MENOR",
    "VS_CP_GROOMING",
    "VS_CP_ACOSO_VIRTUAL",
    "VS_CP_DIFUSION_INTIMIDAD",
    "VS_CP_OTROS",
]

existing = {
    field.name
    for field in client.get_table(A).schema
}

drop_cp = [
    column
    for column in cp_columns
    if column in existing
]

source_select = (
    "* EXCEPT("
    + ", ".join(f"`{column}`" for column in drop_cp)
    + ")"
    if drop_cp
    else "*"
)

cp_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
  {source_select},

  CASE
    WHEN
      (C4P248_1 = 1 AND C4P248C_1 = 1)
      OR (C4P248_2 = 1 AND C4P248C_2 = 1)
    THEN 1
    ELSE 0
  END AS VS_CP_ACOSO_AGRAVADO,

  CASE
    WHEN
      (C4P248_3 = 1 AND C4P248C_3 = 1)
      OR (C4P248_7 = 1 AND C4P248C_7 = 1)
      OR (C4P248_9 = 1 AND C4P248C_9 = 1)
    THEN 1
    ELSE 0
  END AS VS_CP_EXHIBICIONISMO,

  CASE
    WHEN
      (C4P248_4 = 1 AND C4P248C_4 = 1)
      OR (C4P248_5 = 1 AND C4P248C_5 = 1)
      OR (C4P248_6 = 1 AND C4P248C_6 = 1)
      OR (C4P248_8 = 1 AND C4P248C_8 = 1)
    THEN 1
    ELSE 0
  END AS VS_CP_TOCAMIENTOS,

  CASE
    WHEN C4P248_10 = 1
     AND C4P248C_10 = 1
    THEN 1
    ELSE 0
  END AS VS_CP_TENTATIVA_VIOLACION,

  CASE
    WHEN C4P248_11 = 1
     AND C4P248C_11 = 1
    THEN 1
    ELSE 0
  END AS VS_CP_VIOLACION_MENOR,

  CASE
    WHEN C4P248_13 = 1
     AND C4P248C_13 = 1
    THEN 1
    ELSE 0
  END AS VS_CP_GROOMING,

  CASE
    WHEN C4P248_14 = 1
     AND C4P248C_14 = 1
    THEN 1
    ELSE 0
  END AS VS_CP_ACOSO_VIRTUAL,

  CASE
    WHEN
      (C4P248_15 = 1 AND C4P248C_15 = 1)
      OR (C4P248_16 = 1 AND C4P248C_16 = 1)
    THEN 1
    ELSE 0
  END AS VS_CP_DIFUSION_INTIMIDAD,

  CASE
    WHEN C4P248_12 = 1
     AND C4P248C_12 = 1
    THEN 1
    ELSE 0
  END AS VS_CP_OTROS

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_34_codigo_penal.sql").write_text(
    cp_sql,
    encoding="utf-8",
)

client.query(cp_sql).result()

print("Categorías del Código Penal creadas correctamente.")

Categorías del Código Penal creadas correctamente.


In [11]:
# ============================================================
# SPSS 3.4 — P248_01_12M a P248_16_12M
# ============================================================

specific_columns = [
    f"P248_{i:02d}_12M"
    for i in range(1, 17)
]

existing = {
    field.name
    for field in client.get_table(A).schema
}

drop_specific = [
    column
    for column in specific_columns
    if column in existing
]

source_select = (
    "* EXCEPT("
    + ", ".join(f"`{column}`" for column in drop_specific)
    + ")"
    if drop_specific
    else "*"
)

specific_expressions = []

for i in range(1, 17):
    specific_expressions.append(f"""
    CASE
      WHEN `C4P248_{i}` = 1
       AND `C4P248C_{i}` = 1
      THEN 1
      ELSE 0
    END AS `P248_{i:02d}_12M`
    """)

specific_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
  {source_select},
  {",".join(specific_expressions)}

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_34_specific_forms.sql").write_text(
    specific_sql,
    encoding="utf-8",
)

client.query(specific_sql).result()

print("Las 16 formas específicas fueron creadas.")

Las 16 formas específicas fueron creadas.


In [12]:
# ============================================================
# SPSS 3.4 — Grupos de agresores de violencia sexual, 12 meses
#
# Fuente:
# 10_CRS04_3.4 Violencia sexual en adolescentes
#
# Grupos:
# - AggrVS_Familiares_12m: códigos 1–17
# - AggrVS_CAR_12m: códigos 18–21
# - AggrVS_AdultosColegio_12m: códigos 22–24
# - AggrVS_OtraPersona_12m: código 25
# - AggrVS_ParejaExpareja_12m: código 26
# - AggrVS_ParesEscolares_12m: códigos 27–28
#
# Compatible con BigQuery Sandbox:
# CREATE OR REPLACE TABLE, sin UPDATE.
# ============================================================

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

# ------------------------------------------------------------
# 1. Verificar variables fuente
# ------------------------------------------------------------

required_aggressors = ["SEXO"]

for item in range(1, 17):
    required_aggressors.extend([
        f"C4P248_{item}",
        f"C4P248C_{item}",
    ])

    for aggressor_code in range(1, 29):
        required_aggressors.append(
            f"C4P248A_{aggressor_code}_{item}"
        )

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

missing_aggressors = sorted(
    set(required_aggressors) - existing_columns
)

if missing_aggressors:
    raise RuntimeError(
        "No se pueden crear los grupos de agresores. "
        "Faltan variables fuente: "
        + ", ".join(missing_aggressors)
    )

print("Variables fuente de los grupos de agresores verificadas.")


# ------------------------------------------------------------
# 2. Definir códigos de cada grupo
# ------------------------------------------------------------

aggressor_groups = {
    "AggrVS_Familiares_12m": list(range(1, 18)),
    "AggrVS_CAR_12m": [18, 19, 20, 21],
    "AggrVS_AdultosColegio_12m": [22, 23, 24],
    "AggrVS_OtraPersona_12m": [25],
    "AggrVS_ParejaExpareja_12m": [26],
    "AggrVS_ParesEscolares_12m": [27, 28],
}

derived_aggressor_columns = list(aggressor_groups.keys())


# ------------------------------------------------------------
# 3. Preparar reejecución segura
# ------------------------------------------------------------

columns_to_replace = [
    column
    for column in derived_aggressor_columns
    if column in existing_columns
]

if columns_to_replace:
    source_select = (
        "* EXCEPT("
        + ", ".join(
            f"`{column}`"
            for column in columns_to_replace
        )
        + ")"
    )
else:
    source_select = "*"


# ------------------------------------------------------------
# 4. Construir condiciones SQL
# ------------------------------------------------------------

aggressor_expressions = []

for output_variable, aggressor_codes in aggressor_groups.items():

    item_conditions = []

    for item in range(1, 17):

        aggressor_condition = " OR ".join(
            f"`C4P248A_{code}_{item}` = 1"
            for code in aggressor_codes
        )

        item_conditions.append(f"""
        (
          `C4P248_{item}` = 1
          AND `C4P248C_{item}` = 1
          AND (
            {aggressor_condition}
          )
        )
        """)

    any_item_condition = "\nOR\n".join(item_conditions)

    aggressor_expressions.append(f"""
    CASE
      WHEN SEXO IN (1, 2)
       AND (
         {any_item_condition}
       )
      THEN 1
      ELSE 0
    END AS `{output_variable}`
    """)

aggressor_select_sql = ",\n".join(
    aggressor_expressions
)


# ------------------------------------------------------------
# 5. Crear grupos de agresores
# ------------------------------------------------------------

aggressor_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
  {source_select},

  {aggressor_select_sql}

FROM `{A}`
"""

(
    SQL_DIR
    / "stage3_syntax_34_aggressor_groups_12m.sql"
).write_text(
    aggressor_sql,
    encoding="utf-8"
)

print(aggressor_sql)

client.query(aggressor_sql).result()

print(
    "Grupos de agresores de violencia sexual "
    "en los últimos 12 meses creados correctamente."
)


# ------------------------------------------------------------
# 6. Validar dominio y universo
# ------------------------------------------------------------

validation_parts = []

for variable in derived_aggressor_columns:

    validation_sql = f"""
    SELECT
      '{variable}' AS variable,

      COUNT(*) AS total_rows,

      COUNTIF(
        `{variable}` IS NULL
        OR `{variable}` NOT IN (0, 1)
      ) AS invalid_values,

      COUNTIF(`{variable}` = 0) AS zero_values,

      COUNTIF(`{variable}` = 1) AS one_values

    FROM `{A}`
    """

    validation_parts.append(
        client.query(validation_sql)
        .result()
        .to_dataframe()
    )

aggressor_validation = pd.concat(
    validation_parts,
    ignore_index=True
)

aggressor_validation.to_csv(
    LOG_DIR
    / "stage3_syntax_34_aggressor_groups_12m_validation.csv",
    index=False
)

display(aggressor_validation)

if (
    aggressor_validation["total_rows"] != EXPECTED_ROWS
).any():
    raise RuntimeError(
        "Algún grupo de agresores alteró el universo analítico."
    )

if (
    aggressor_validation["invalid_values"] > 0
).any():
    raise RuntimeError(
        "Algún grupo de agresores contiene valores fuera de 0/1."
    )

print(
    "Grupos de agresores validados: "
    "18,807 filas y dominio exclusivamente 0/1."
)

Variables fuente de los grupos de agresores verificadas.

CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

SELECT
  * EXCEPT(`AggrVS_Familiares_12m`, `AggrVS_CAR_12m`, `AggrVS_AdultosColegio_12m`, `AggrVS_OtraPersona_12m`, `AggrVS_ParejaExpareja_12m`, `AggrVS_ParesEscolares_12m`),

  
    CASE
      WHEN SEXO IN (1, 2)
       AND (
         
        (
          `C4P248_1` = 1
          AND `C4P248C_1` = 1
          AND (
            `C4P248A_1_1` = 1 OR `C4P248A_2_1` = 1 OR `C4P248A_3_1` = 1 OR `C4P248A_4_1` = 1 OR `C4P248A_5_1` = 1 OR `C4P248A_6_1` = 1 OR `C4P248A_7_1` = 1 OR `C4P248A_8_1` = 1 OR `C4P248A_9_1` = 1 OR `C4P248A_10_1` = 1 OR `C4P248A_11_1` = 1 OR `C4P248A_12_1` = 1 OR `C4P248A_13_1` = 1 OR `C4P248A_14_1` = 1 OR `C4P248A_15_1` = 1 OR `C4P248A_16_1` = 1 OR `C4P248A_17_1` = 1
          )
        )
        
OR

        (
          `C4P248_2` = 1
          AND `C4P248C_2` = 1
          AND (
            `C4P248A_1_2` = 1

,variable,total_rows,invalid_values,zero_values,one_values
0,AggrVS_Familiares_12m,18807,0,18514,293
1,AggrVS_CAR_12m,18807,0,18804,3
2,AggrVS_AdultosColegio_12m,18807,0,18782,25
3,AggrVS_OtraPersona_12m,18807,0,17524,1283
4,AggrVS_ParejaExpareja_12m,18807,0,18684,123
5,AggrVS_ParesEscolares_12m,18807,0,16657,2150


Grupos de agresores validados: 18,807 filas y dominio exclusivamente 0/1.
